In [1]:
import argparse, yaml, config, os, torch, collections, cv2, sys, json
from multiprocessing.pool import Pool
from functools import partial
from multiprocessing import Manager
import pandas as pd
from tqdm import tqdm
import numpy as np
class SuppressOutput:
    def __enter__(self):
        self._original_stderr = sys.stderr
        sys.stderr = open(os.devnull, 'w')
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stderr.close()
        sys.stderr = self._original_stderr

with SuppressOutput():
    from mediapipe.python.solutions import face_mesh

from albumentations import Compose, RandomBrightnessContrast, \
    HorizontalFlip, FancyPCA, HueSaturationValue, OneOf, ToGray, \
    ShiftScaleRotate, ImageCompression, PadIfNeeded, GaussNoise, GaussianBlur, \
    LongestMaxSize, KeypointParams

os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["GLOG_minloglevel"] = "3"   # 0=INFO, 1=WARNING, 2=ERROR, 3=FATAL (2->3으로 변경)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # (2->3으로 변경)

ROOT_DIR = config.ROOT_DIR
DATA_DIR = os.path.join(ROOT_DIR, "data")
DF40_DIR = os.path.join(DATA_DIR, "DF40")
MODELS_PATH = "models"
METADATA_PATH = os.path.join(DATA_DIR, "dataset_json")

In [2]:
json_lst = os.listdir(METADATA_PATH)
json_lst[0]

'blendface_cdf.json'

In [3]:
json_lst

['blendface_cdf.json',
 'blendface_ff.json',
 'CollabDiff.json',
 'danet_cdf.json',
 'danet_ff.json',
 'ddim_cdf.json',
 'ddim_ff.json',
 'deepfacelab.json',
 'DF40_all.json',
 'DiT_cdf.json',
 'DiT_ff.json',
 'e4e_cdf.json',
 'e4e_ff.json',
 'e4s_cdf.json',
 'e4s_ff.json',
 'EFSAll_cdf.json',
 'EFSAll_ff.json',
 'facedancer_cdf.json',
 'facedancer_ff.json',
 'faceswap_cdf.json',
 'faceswap_ff.json',
 'facevid2vid_cdf.json',
 'facevid2vid_ff.json',
 'fomm_cdf.json',
 'fomm_ff.json',
 'FRAll_cdf.json',
 'FRAll_ff.json',
 'FSAll_cdf.json',
 'FSAll_ff.json',
 'fsgan_cdf.json',
 'fsgan_ff.json',
 'heygen.json',
 'hyperreenact_cdf.json',
 'hyperreenact_ff.json',
 'inswap_cdf.json',
 'inswap_ff.json',
 'lia_cdf.json',
 'lia_ff.json',
 'mcnet_cdf.json',
 'mcnet_ff.json',
 'MidJourney.json',
 'mobileswap_cdf.json',
 'mobileswap_ff.json',
 'MRAA_cdf.json',
 'MRAA_ff.json',
 'one_shot_free_cdf.json',
 'one_shot_free_ff.json',
 'pirender_cdf.json',
 'pirender_ff.json',
 'pixart_cdf.json',
 'pixar

In [4]:
with open(os.path.join(METADATA_PATH, "blendface_cdf.json"), 'r') as f:
    json_data_cdf = json.load(f)

In [5]:
with open(os.path.join(METADATA_PATH, "DiT_ff.json"), 'r') as f:
    json_data_ff = json.load(f)

In [6]:
json_data_ff['DiT_ff'].keys()

dict_keys(['DiT_Real', 'DiT_Fake'])

In [8]:
json_data_ff['DiT_ff']['DiT_Fake']['train']

{'446': {'label': 'DiT_Fake',
  'frames': ['deepfakes_detection_datasets/DF40/DiT/ff/446/13644.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/3581.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/10127.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/2229.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/6064.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/5866.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/12078.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/19987.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/15434.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/18677.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/17210.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/5862.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/17322.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/13065.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/18357.png',
   'deepfakes_detection_datasets/DF40/DiT/ff/446/1

In [9]:
with open(os.path.join(METADATA_PATH, "pixart_ff.json"), 'r') as f:
    json_data2 = json.load(f)

In [10]:
json_data2['pixart_ff'].keys()

dict_keys(['pixart_Real', 'pixart_Fake'])

In [12]:
json_data2['pixart_ff']['pixart_Fake']['test']

{'078': {'label': 'pixart_Fake',
  'frames': ['deepfakes_detection_datasets/DF40/pixart/ff/078/825_493.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/313_092.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/857_577.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/646_396.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/906_025.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/937_073.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/736_279.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/003_263.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/941_031.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/816_343.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/420_544.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/385_033.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/376_447.png',
   'deepfakes_detection_datasets/DF40/pixart/ff/078/475_097.png',
   'deepfakes_detection_datasets/

In [35]:
from pathlib import Path

for d in json_data2['danet_ff']['danet_Real']['train']:
    print(len(json_data2['danet_ff']['danet_Real']['train'][d]['frames']))

32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
31
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
31
32
32
32
32
31
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
29
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
31
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
31
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
3

In [14]:
with open(os.path.join(METADATA_PATH, "RDDM_ff.json"), 'r') as f:
    json_data3 = json.load(f)

In [17]:
json_data3['rddm_ff'].keys()

dict_keys(['rddm_Real', 'rddm_Fake'])